In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# GROW 固定单侧写入裕量：生成与完整媒体验证

选择GPU后点击“全部运行”。本次固定新生成4case × OFF/A/B=12视频，
随后全部12都执行terminal/floatRGB/RGB8/MP4四层验证，共48层。
不因终态恢复失败跳过媒体；生成本身失败保留缺失记录，不补跑或筛样本。

唯一方法改动是写入目标：L=.5*sum(relu(.5-b*d)^2)，其中d仍是固定23对相邻时间
正交差分。保持16位/92票hard读出、channel0频率2..9、eta.1、控制30..49、
原生50步UniPC和原内容种子。正确且超过现有.5目标的系数不再被拉回目标值。
.5是写入裕量，不是新增检测阈值；没有soft混合、ECC、pair筛选或强度扫描。

先完成各case独立生成子进程，再启动独立VAE媒体子进程，Transformer不留在媒体显存。
固定1200 Transformer、600live step、160局部梯度、160shadow，
另12VAE decode、36encode、12MP4保存/回读。计数按生成/媒体分开并汇总。
每层固定8marked/128位，OFF单列，失败不缩分母；质量仅相对诊断。
请人工查看提示词内容、伪影、闪烁、运动和连贯性；PSNR没有硬门槛。

独立输出 `MyDrive/Video-WM/GROWTemporalMargin/grow_temporal_margin_<UTC>`，
generation与media分别保存配置、哈希、张量、结果及子日志，父级source.zip/launcher.log。
媒体中间RGB约6.4GB另加latent/MP4，请保留结果以核验，但不新增设备硬门槛。
现有CPU/Fake只能证明工程路径，真实成功、质量和显存待本次用户运行。
固定源码 SHA：52dd860102f2bbaa3236e05289addc1b74b17942。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = '52dd860102f2bbaa3236e05289addc1b74b17942'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'grow_temporal_margin_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/GROWTemporalMargin') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.grow_temporal_margin_full', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({'status': result['status'], 'video_denominator': result['video_denominator'], 'recovery_summary': result.get('recovery_summary'), 'stage_status': {k:v['status'] for k,v in result['stages'].items()}, 'fixed_calls': result['fixed_calls'], 'actual_calls_observed': result.get('actual_calls_observed'), 'layer_denominator':result['layer_denominator']}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
